In [438]:
import requests
from bs4 import BeautifulSoup
import os
import numpy as np
import time
import re
import unicodedata

In [ ]:
def polite_get(url, min_delay=0.5, max_delay=1.5, max_retries=5):
    time.sleep(np.random.uniform(min_delay, max_delay))
    for attempt in range(max_retries):
        r = session.get(url)
        if r.status_code == 200:
            return r
        if r.status_code == 429:
            ra = r.headers.get("Retry-After")
            wait = float(ra) if ra else min(2**attempt, 60)
            time.sleep(wait)
            continue
        if 500 <= r.status_code < 600:
            time.sleep(min(2**attempt, 60))
            continue
        r.raise_for_status()
    raise RuntimeError("Max retries exceeded")


def scrape_tale(r):
    soup = BeautifulSoup(r.text, "html.parser")
    title = soup.find("title").text.strip()
    date = soup.select_one("time")["datetime"]

    topics = soup.select("div.speech-topics")[0].find_all("a")
    topic_categories = []
    topic_names = []
    for topic in topics:
        topic_category = topic.attrs['title'].split("</span>")[0].split(">")[-1].strip()
        topic_categories.append(topic_category)
        topic_names.append(topic.text.strip())

    text_base = soup.find("div", class_="speech-article-content")
    text = ""
    if text_base is None:
        return "Error_text_not_found", date, topic_categories, topic_names, "Text not found"
    for node in text_base.descendants:
        if node.name is None:
            text += node.strip()
            text += "\n"

    return title, date, topic_categories, topic_names, text


def sanitize_filename(name, replacement="_"):
    name = unicodedata.normalize("NFKD", name)
    name = "".join(ch for ch in name if not unicodedata.combining(ch))
    name = re.sub(r'[<>:"/\\|?*\x00-\x1f]', replacement, name)
    name = re.sub(r"\s+", replacement, name).strip()
    name = name.strip(" .")
    name = re.sub(rf"{re.escape(replacement)}+", replacement, name)
    return name or "untitled"

def save_tale(title, date, topic_categories, topic_names, text):
    title = sanitize_filename(title)
    filename = "Speeches/" + title + ".txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(f"Title: {title}\n")
        f.write(f"Date: {date}\n")
        for i, topic in enumerate(topic_categories):
            f.write(f"{topic}: {topic_names[i]}\n")
        f.write("\n")
        f.write("MAIN_BODY_OF_SPEECH:\n")
        f.write(text)

In [444]:
sanitize_filename(title)

"Lysias'_tale_mod_forsøget_pa_at_ophæve_den_fædrene_forfatning_403_2_f.v.t"

In [396]:
base_url = "https://www.dansketaler.dk"
speeches_url = np.genfromtxt("T_speech_link.txt", dtype=str, delimiter="\n")

In [483]:
session = requests.Session()
session.headers.update({"User-Agent": "KU MachineLearning2026 FinalProject Bot/1.0"})

for speech_url in speeches_url[3258:]:
    full_url = base_url + speech_url
    print(f"Processing: {full_url}")
    r = polite_get(full_url)
    title, date, topic_categories, topic_names, text = scrape_tale(r)
    save_tale(title, date, topic_categories, topic_names, text)

session.close()

Processing: https://www.dansketaler.dk/tale/pia-soltofts-praediken-sidste-sondag-i-kirkearet
Processing: https://www.dansketaler.dk/tale/dorrit-willumsens-tale-ved-modtagelsen-af-selskabets-medalje
Processing: https://www.dansketaler.dk/tale/michael-vindfeldts-tale-ved-jan-e-jorgensens-25-ars-jubilaeum
Processing: https://www.dansketaler.dk/tale/esben-bjoern-salmonsens-tale-ved-koebenhavns-universitets-aarsfest
Processing: https://www.dansketaler.dk/tale/henrik-c-wegeners-tale-ved-koebenhavns-universitets-aarsfes
Processing: https://www.dansketaler.dk/tale/merete-eldrups-tale-ved-koebenhavns-universitets-aarsfest-2022
Processing: https://www.dansketaler.dk/tale/stine-egedes-tale-forste-sondag-i-advent-2022
Processing: https://www.dansketaler.dk/tale/anne-marie-mais-tale-ved-overraekkelsen-af-selskabets-medalje
Processing: https://www.dansketaler.dk/tale/olga-ravns-tale-til-lone-aburas-ved-overraekkelsen-af-otto-gelsteds-mindelegat
Processing: https://www.dansketaler.dk/tale/lars-boberg

In [482]:
for i, speech in enumerate(speeches_url):
    if speech_url == speech:
        print(i, speech)

3257 /tale/jakob-ellemann-jensens-tale-ved-venstres-landsmoede-2022


In [457]:
soup = BeautifulSoup(r.text, "html.parser")
soup

<!DOCTYPE html>

<html lang="da">
<head>
<title>Kaj Mogensens prædiken 5. søndag efter påske</title>
<meta content='Denne prædiken blev holdt til en "online-andagt" pga. coronanedlukningen af kirkerne i foråret 2020.PrædikentekstJohannesevangeliet kapitel 17, ver...' name="description"/>
<meta content="width=device-width, initial-scale=1" name="viewport"/>
<meta content="/msg/cookie-popup" name="site-msg"/>
<link href="/favicon-32x32.png" rel="icon" sizes="32x32" type="image/png"/>
<link href="/favicon-16x16.png" rel="icon" sizes="16x16" type="image/png"/>
<link href="/favicon-196x196.png" rel="apple-touch-icon"/>
<link href="/assets/application-66bb33b50ea698d286a63cc5875b53ece42854d882edcc770457d37e071d724b.css" rel="stylesheet">
<style>
      :root {
        --sans: Montserrat, sans-serif;
        --wght-400: 400;
        --wght-500: 500;
        --wght-600: 600;
        --wght-700: 700;
        --letter-spacing: normal;

        --exercise: #f0ecc4;
        --lesson: #d9d6c6;
     

In [461]:
soup.find("div", class_="speech-article-content") is not None

False

In [337]:
url = "https://www.dansketaler.dk/tale/mathilde-roer-nielsens-praediken-1-sondag-efter-helligtrekonger"  # konkret tale
#title, org, date, text = scrape_tale(url)
#save_tale(title, org, date, text)

In [338]:
r = polite_get(url)
soup = BeautifulSoup(r.text, "html.parser")

In [370]:
topic = soup.select("div.speech-topics")[0].find_all("a")[0].attrs["title"]
topic.split("</span>")[0].split(">")[-1].strip()

'Søndag Efter Helligtrekonger'

In [381]:
title, date, topic_categories, topic_names, text = scrape_tale(r)
save_tale(title, date, topic_categories, topic_names, text)

0 Søndag Efter Helligtrekonger
1 Første Tekstrække
2 Talegenre
3 Historisk kontekst


In [293]:
for node in text.descendants:
    if node.name is None:
        text = node.strip()
        print(text)



Rigtig mange af mine taler starter med at sætte en tyk streg under, at vi står i en velfærdskrise.
At vi mangler medarbejdere.
I dag starter jeg et andet sted.
For I ved – om nogen – hvilken krise vores velfærd står overfor.
I ser den udfolde sig i jeres daglige arbejde.
I løber hurtigere og hurtigere. Og får færre kollegaer.
…
Alligevel står I op hver dag og gør en forskel.
Knokler for at få enderne til at mødes.
For at møde de mennesker, som I gør en forskel for hver dag.
I hjemmeplejen, på hospitaler og i sygeplejen.
Derfor vil jeg starte min tale i dag med at sige tak.
TAK for at I kæmper og gør københavnernes liv bedre.
Og glædelig 1. maj. Rigtig glædelig kampdag.
Heldigvis har regeringen ikke kastet sig over denne ”fridag” endnu.
Men det er jo heller ikke en fridag.
Det er en kampdag. Og der er stadig meget at kæmpe for.
Min helt store kamp som sundheds- og omsorgsborgmester i København er at sikre fastholdelse og rekruttering af medarbejdere.
Det er en kamp om velfærdssamfundet

In [ ]:
links = {}
for year in range(1950, 2024):
    r = requests.get(f"{base_url}?year={year}")
    soup = BeautifulSoup(r.text, "html.parser")
    for speech in soup.select("a"):
        if "href" in speech.attrs:
            links[speech.attrs["href"]] = year

In [223]:
r = requests.get("https://www.dansketaler.dk/praedikener/tale?year=2025")
soup = BeautifulSoup(r.text, "html.parser")

In [220]:
soup.select("a.current-speech")[0]["href"]

'/tale/lysias-tale-mod-forsoeget-paa-at-ophaeve-den-faedrene-forfatning-403-2-f-v-t'

In [36]:
links = []
for speech in soup.select("a"):
    if "href" in speech.attrs:
        links.append(speech.attrs["href"])

In [191]:
soup.select("div.current-speeches-container")[0]["data-url"]

'/tale/1950/1'

In [234]:
decades = np.genfromtxt("P_decade_link.txt", dtype=str)
years = np.genfromtxt("P_year_link.txt", dtype=str)
months = np.genfromtxt("P_month_link.txt", dtype=str)
base_url = "https://www.dansketaler.dk"

In [215]:
base_url + months[1]

'https://www.dansketaler.dk/tale/-403/0'

In [235]:
session = requests.Session()
session.headers.update({"User-Agent": "KU MachineLearning2026 FinalProject Bot/1.0"})
taler = []

for month in months:
    r = polite_get(base_url + month)
    soup = BeautifulSoup(r.text, "html.parser")
    for tale in soup.select("a.current-speech"):
        taler.append(tale["href"])

np.savetxt("P_speech_link.txt", taler, fmt="%s")
session.close()

In [233]:
session = requests.Session()
session.headers.update({"User-Agent": "KU MachineLearning2026 FinalProject Bot/1.0"})
months = []

for year in years:
    r = polite_get(base_url + year)
    soup = BeautifulSoup(r.text, "html.parser")
    for month in soup.select("div.current-speeches-container"):
        months.append(month["data-url"])

np.savetxt("P_month_link.txt", months, fmt="%s")
session.close()

In [199]:
soup.select("div.current-speeches-container")

[<div class="current-speeches-container" data-url="/tale/-428/0">
 </div>]

In [229]:
session = requests.Session()
session.headers.update({"User-Agent": "KU MachineLearning2026 FinalProject Bot/1.0"})
years = []
for decade in decades:
    r = polite_get(base_url + decade)
    soup = BeautifulSoup(r.text, "html.parser")
    for year in soup.select("div.decade-pagination")[0].select("a"):
        years.append(year["href"])
print(years)
session.close()

['/praedikener/tale?year=1515', '/praedikener/tale?year=1539', '/praedikener/tale?year=1555', '/praedikener/tale?year=1565', '/praedikener/tale?year=1571', '/praedikener/tale?year=1574', '/praedikener/tale?year=1588', '/praedikener/tale?year=1591', '/praedikener/tale?year=1621', '/praedikener/tale?year=1635', '/praedikener/tale?year=1638', '/praedikener/tale?year=1639', '/praedikener/tale?year=1657', '/praedikener/tale?year=1668', '/praedikener/tale?year=1688', '/praedikener/tale?year=1730', '/praedikener/tale?year=1739', '/praedikener/tale?year=1747', '/praedikener/tale?year=1760', '/praedikener/tale?year=1765', '/praedikener/tale?year=1776', '/praedikener/tale?year=1781', '/praedikener/tale?year=1782', '/praedikener/tale?year=1788', '/praedikener/tale?year=1794', '/praedikener/tale?year=1803', '/praedikener/tale?year=1810', '/praedikener/tale?year=1812', '/praedikener/tale?year=1813', '/praedikener/tale?year=1815', '/praedikener/tale?year=1820', '/praedikener/tale?year=1825', '/praed

In [230]:
np.savetxt("P_year_link.txt", years, fmt="%s")

In [225]:
decades = []
for link in soup.select("li"):
    if not link.select("a"):
        continue
    link_href = link.select("a")[0]["href"]
    if "tale?" in link_href:
        decades.append(link_href)
print(decades)


['/praedikener/tale?year=1515', '/praedikener/tale?year=1539', '/praedikener/tale?year=1555', '/praedikener/tale?year=1565', '/praedikener/tale?year=1571', '/praedikener/tale?year=1588', '/praedikener/tale?year=1591', '/praedikener/tale?year=1621', '/praedikener/tale?year=1635', '/praedikener/tale?year=1657', '/praedikener/tale?year=1668', '/praedikener/tale?year=1688', '/praedikener/tale?year=1730', '/praedikener/tale?year=1747', '/praedikener/tale?year=1760', '/praedikener/tale?year=1776', '/praedikener/tale?year=1781', '/praedikener/tale?year=1794', '/praedikener/tale?year=1803', '/praedikener/tale?year=1810', '/praedikener/tale?year=1820', '/praedikener/tale?year=1840', '/praedikener/tale?year=1850', '/praedikener/tale?year=1860', '/praedikener/tale?year=1871', '/praedikener/tale?year=1881', '/praedikener/tale?year=1892', '/praedikener/tale?year=1907', '/praedikener/tale?year=1918', '/praedikener/tale?year=1926', '/praedikener/tale?year=1934', '/praedikener/tale?year=1940', '/praed

In [226]:
np.savetxt("P_decade_link.txt", decades, fmt="%s")

In [119]:
soup.select("li")[12].select("a")#[0]["href"]

[]

In [127]:
if not soup.select("li")[5].select("a"):
    print("No link found")

In [236]:
Pr = np.genfromtxt("P_speech_link.txt", dtype=str)
Tl = np.genfromtxt("T_speech_link.txt", dtype=str)

In [237]:
for speech in Pr:
    if speech in Tl:
        continue
    print(speech)

In [239]:
Tl_not_in_Pr = [speech for speech in Tl if speech not in Pr]

In [245]:
len(Tl_not_in_Pr)

4518